In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q torch_geometric
!pip install -q class_resolver
!pip3 install pymatting

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.4 MB/s eta 0:00:00


In [4]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss
from scipy import sparse
from scipy.sparse.linalg import eigsh
from torch.utils.data import TensorDataset, DataLoader, Subset
import random

In [5]:
data = np.load('/content/drive/MyDrive/TejaswiAbburi_va797/Dataset/Medmnist_data/pneumoniamnist_224.npz', allow_pickle=True)

all_images = np.concatenate([data['train_images'], data['val_images'], data['test_images']], axis=0)
all_labels = np.concatenate([data['train_labels'], data['val_labels'], data['test_labels']], axis=0).squeeze()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),  # grayscale → 3-channel
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

images = torch.stack([transform(img) for img in all_images])
labels = torch.tensor(all_labels).long()

In [6]:
dataset = TensorDataset(images, labels)
class0_indices = [i for i in range(len(labels)) if labels[i] == 0]
class1_indices = [i for i in range(len(labels)) if labels[i] == 1]

random.seed(42)
sampled_class0 = random.sample(class0_indices, min(2000, len(class0_indices)))
sampled_class1 = random.sample(class1_indices, min(2000, len(class1_indices)))
combined_indices = sampled_class0 + sampled_class1
random.shuffle(combined_indices)

final_dataset = Subset(dataset, combined_indices)
final_loader = DataLoader(final_dataset, batch_size=64, shuffle=False)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
vit = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16')  # DINO ViT-B/16
vit.eval().to(device)

vit_feats = []
y_list = []

with torch.no_grad():
    for imgs, lbls in final_loader:
        imgs = imgs.to(device)
        feats = vit(imgs)  # (N, 768)
        vit_feats.append(feats.cpu())
        y_list.extend(lbls.cpu().tolist())

F = torch.cat(vit_feats, dim=0).numpy().astype(np.float32)
y_labels = np.array(y_list).astype(np.int64)

print("Feature shape:", F.shape)
print("Label shape:", y_labels.shape)

Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_vitbase16_pretrain/dino_vitbase16_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_vitbase16_pretrain.pth


100%|██████████| 327M/327M [00:03<00:00, 102MB/s] 


Feature shape: (3583, 768)
Label shape: (3583,)


In [8]:
def tokencut_on_features(F_array, alpha=1e-6):
    N, D = F_array.shape

    # Normalize features row-wise
    norms = np.linalg.norm(F_array, axis=1, keepdims=True) + 1e-10
    F_norm = F_array / norms

    # Cosine similarity matrix
    W = np.dot(F_norm, F_norm.T)
    W = W + alpha

    # Normalized Laplacian
    d = np.sum(W, axis=1)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(d + 1e-10))
    L = np.eye(N) - d_inv_sqrt @ W @ d_inv_sqrt

    L_sparse = sparse.csr_matrix(L)

    # Fiedler vector (2nd smallest eigenvector)
    vals, vecs = eigsh(L_sparse, k=2, which='SM')
    fiedler = vecs[:, 1]

    # Threshold by mean
    threshold = fiedler.mean()
    labels = (fiedler > threshold).astype(np.int64)

    return labels, fiedler

labels, scores = tokencut_on_features(F)

In [9]:
y_pred = labels
acc = accuracy_score(y_labels, y_pred)
inv_acc = accuracy_score(y_labels, 1 - y_pred)
if inv_acc > acc:
    y_pred = 1 - y_pred
    acc = inv_acc

prec = precision_score(y_labels, y_pred)
rec = recall_score(y_labels, y_pred)
f1 = f1_score(y_labels, y_pred)

# Normalize fiedler scores for logloss
probs = (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)
logloss = log_loss(y_labels, probs)

print("===== TokenCut Results (PneumoniaMNIST) =====")
print("Accuracy Score:", acc)
print("Precision Score:", prec)
print("Recall Score:", rec)
print("F1 Score:", f1)
print("Log Loss:", logloss)

===== TokenCut Results (PneumoniaMNIST) =====
Accuracy Score: 0.8847334635780073
Precision Score: 0.9542072123640527
Recall Score: 0.8335
F1 Score: 0.8897784894582332
Log Loss: 0.43115661973723046


In [10]:
print(y_pred)

[0 1 0 ... 0 0 1]


In [11]:
print(y_labels)

[0 1 0 ... 0 0 1]


In [12]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss, normalized_mutual_info_score

num_runs = 10

acc_scores, prec_scores, rec_scores, f1_scores, log_losses, nmi_scores = [], [], [], [], [], []

for run in range(num_runs):
    print(f"\n--- Run {run+1}/{num_runs} ---")
    np.random.seed(run)
    torch.manual_seed(run)

    y_pred, scores = tokencut_on_features(F)

    acc = accuracy_score(y_labels, y_pred)
    inv_acc = accuracy_score(y_labels, 1 - y_pred)
    if inv_acc > acc:
        y_pred = 1 - y_pred
        acc = inv_acc

    prec = precision_score(y_labels, y_pred, zero_division=0)
    rec = recall_score(y_labels, y_pred, zero_division=0)
    f1 = f1_score(y_labels, y_pred, zero_division=0)
    nmi = normalized_mutual_info_score(y_labels, y_pred)

    probs = (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)
    logloss = log_loss(y_labels, probs)

    acc_scores.append(acc)
    prec_scores.append(prec)
    rec_scores.append(rec)
    f1_scores.append(f1)
    log_losses.append(logloss)
    nmi_scores.append(nmi)

    print(f"Run {run+1} | Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | "
          f"F1: {f1:.4f} | LogLoss: {logloss:.4f} | NMI: {nmi:.4f}")

print("\n================ FINAL SUMMARY ================\n")
print(f"{'Metric':>15} | {'Mean':>10} ± {'Std':<10}")
print("-" * 50)
print(f"{'Accuracy':>15} | {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
print(f"{'Precision':>15} | {np.mean(prec_scores):.4f} ± {np.std(prec_scores):.4f}")
print(f"{'Recall':>15} | {np.mean(rec_scores):.4f} ± {np.std(rec_scores):.4f}")
print(f"{'F1 Score':>15} | {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"{'Log Loss':>15} | {np.mean(log_losses):.4f} ± {np.std(log_losses):.4f}")
print(f"{'NMI Score':>15} | {np.mean(nmi_scores):.4f} ± {np.std(nmi_scores):.4f}")


--- Run 1/10 ---
Run 1 | Acc: 0.8847 | Prec: 0.9542 | Rec: 0.8335 | F1: 0.8898 | LogLoss: 1.2227 | NMI: 0.5120

--- Run 2/10 ---
Run 2 | Acc: 0.8847 | Prec: 0.9542 | Rec: 0.8335 | F1: 0.8898 | LogLoss: 0.4312 | NMI: 0.5120

--- Run 3/10 ---
Run 3 | Acc: 0.8847 | Prec: 0.9542 | Rec: 0.8335 | F1: 0.8898 | LogLoss: 1.2227 | NMI: 0.5120

--- Run 4/10 ---
Run 4 | Acc: 0.8847 | Prec: 0.9542 | Rec: 0.8335 | F1: 0.8898 | LogLoss: 0.4312 | NMI: 0.5120

--- Run 5/10 ---
Run 5 | Acc: 0.8847 | Prec: 0.9542 | Rec: 0.8335 | F1: 0.8898 | LogLoss: 1.2227 | NMI: 0.5120

--- Run 6/10 ---
Run 6 | Acc: 0.8847 | Prec: 0.9542 | Rec: 0.8335 | F1: 0.8898 | LogLoss: 0.4312 | NMI: 0.5120

--- Run 7/10 ---
Run 7 | Acc: 0.8847 | Prec: 0.9542 | Rec: 0.8335 | F1: 0.8898 | LogLoss: 0.4312 | NMI: 0.5120

--- Run 8/10 ---
Run 8 | Acc: 0.8847 | Prec: 0.9542 | Rec: 0.8335 | F1: 0.8898 | LogLoss: 1.2227 | NMI: 0.5120

--- Run 9/10 ---
Run 9 | Acc: 0.8847 | Prec: 0.9542 | Rec: 0.8335 | F1: 0.8898 | LogLoss: 0.4312 | NMI:

In [13]:
max_probability_value = np.max(probs)
print("The maximum probability value in the array is:", max_probability_value)

The maximum probability value in the array is: 0.9999999986938349
